# Benchmark de Modelos de Forecasting: Predição de Dengue

Este notebook executa o benchmark de backtesting com rolling origin para comparar os modelos de forecasting disponíveis no projeto.

**Modelos avaliados:**
- SARIMAX (baseline estatístico)
- LightGBM
- XGBoost
- CatBoost
- RandomForest
- Ridge

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from dengue_forecast.evaluate import run_backtest
from dengue_forecast.models import (
    SarimaxForecaster,
    catboost_forecaster,
    lgbm_forecaster,
    randomforest_forecaster,
    ridge_forecaster,
    xgboost_forecaster,
)

sns.set_theme(style='whitegrid', palette='colorblind')

## 1. Carregamento dos dados

In [ ]:
series = pd.read_csv(
    '../data/processed/dengue_monthly_sao.csv',
    index_col='date',
    parse_dates=['date'],
)['value'].asfreq('MS')

print(f'Série: {series.name}')
print(f'Período: {series.index.min().date()} a {series.index.max().date()}')
print(f'Pontos: {len(series)}')
print(f'Total de casos: {int(series.sum()):,}')
series.plot(figsize=(14, 4), title='Casos Mensais de Dengue — São Paulo (2010–2024)', ylabel='Casos')
plt.tight_layout()
plt.show()

## 2. Configuração dos modelos

In [ ]:
HORIZON = 12       # 12 meses de horizonte de previsão
MIN_TRAIN = 48     # 48 meses de treino mínimo
LAGS = 24          # 24 lags para modelos de ML

models = [
    SarimaxForecaster(order=(1, 1, 1), seasonal_order=(1, 1, 0, 12)),
    lgbm_forecaster(lags=LAGS),
    xgboost_forecaster(lags=LAGS),
    catboost_forecaster(lags=LAGS),
    randomforest_forecaster(lags=LAGS),
    ridge_forecaster(lags=LAGS),
]

print(f'Modelos a avaliar: {[m.name for m in models]}')

## 3. Execução do backtesting

In [ ]:
metrics_rows = []
preds_frames = []

for model in models:
    metric_row, pred_df = run_backtest(
        series=series,
        model=model,
        horizon=HORIZON,
        min_train_size=MIN_TRAIN,
    )
    if metric_row is not None:
        metrics_rows.append(metric_row)
    if not pred_df.empty:
        preds_frames.append(pred_df)

metrics_df = pd.DataFrame(metrics_rows).sort_values('smape')
preds_df = pd.concat(preds_frames, ignore_index=True)

## 4. Resultados

In [ ]:
metrics_df.style.highlight_min(subset=['mae', 'rmse', 'smape'], color='lightgreen')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics_info = [
    ('mae', 'MAE', 'Mean Absolute Error'),
    ('rmse', 'RMSE', 'Root Mean Squared Error'),
    ('smape', 'sMAPE (%)', 'Symmetric MAPE'),
]

for ax, (col, ylabel, title) in zip(axes, metrics_info):
    colors = ['gold' if i == 0 else 'steelblue' for i in range(len(metrics_df))]
    ax.barh(metrics_df['model'], metrics_df[col], color=colors)
    ax.set_xlabel(ylabel)
    ax.set_title(title, fontweight='bold')
    ax.invert_yaxis()

plt.suptitle('Benchmark de Modelos — São Paulo (2010–2024)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Previsões vs. Valores Reais

In [ ]:
best_model = metrics_df.iloc[0]['model']
sub = preds_df[preds_df['model'] == best_model].sort_values('date')

plt.figure(figsize=(14, 5))
plt.plot(sub['date'], sub['y_true'], label='Real', color='black', linewidth=1.5)
plt.plot(sub['date'], sub['y_pred'], label=f'Previsto ({best_model})', color='steelblue', linewidth=1, linestyle='--')
plt.title(f'Melhor Modelo: {best_model.upper()} — São Paulo', fontweight='bold')
plt.ylabel('Casos mensais')
plt.legend()
plt.tight_layout()
plt.show()